In [3]:
!pip install openai pandas

  Using cached openai-2.7.1-py3-none-any.whl.metadata (29 kB)
  Using cached pandas-2.3.3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (91 kB)
  Using cached anyio-4.11.0-py3-none-any.whl.metadata (4.1 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jiter-0.11.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.2 kB)
  Using cached pydantic-2.12.4-py3-none-any.whl.metadata (89 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached certifi-2025.10.5-py3-none-any.whl.metadata (2.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annota

In [6]:
!pip install python-dotenv

In [1]:
import json
from openai import OpenAI
import pandas as pd
from IPython.display import Image, display

In [2]:
import os
import logging
logging.basicConfig(
    format='%(asctime)s : %(levelname)s - %(message)s',
    level=logging.INFO
)
from dotenv import load_dotenv
load_dotenv()


True

In [ ]:
OPENAI_API_KEY=os.getenv('OPENAI_API_KEY')


In [4]:
client=OpenAI(
    api_key=OPENAI_API_KEY
)



In [5]:
response=client.responses.create(
    model='gpt-5-mini',
    input='Write one sentence bedtime story about Korean 2015 drama series Reply 1988'
)
print(response.output_text)

KeyboardInterrupt: 

In [5]:
df=pd.read_csv('/home/aakash/NIC/cdl-updated/nic-metadata-cleaning/data/original/metadata_transformation.csv')
df.head()

,dcterms:title,dcterms:description,dcat:theme,dcterms:created,dcterms:issued,dcterms:modified,dcterms:type,dcat:ByteSize,dcat:mediaType,dcterms:relation,...,dcat:inSeries,No mapping available,No mapping available.1,No mapping available.2,No mapping available.3,No mapping available.4,No mapping available.5,No mapping available.6,No mapping available.7,No mapping available.8
0,Daily Wholesale Mandi Market Prices of Agricul...,The data refers to prices of variety wise agri...,"Agriculture, Agricultural Marketing",2024-05-21,2024-06-02,2025-10-15,Dataset,1464,text/csv,http://agmarknet.gov.in,...,Intentionally Kept Blank,167942,394032,23 (indicative),TRUE,1.0,6622308.0,1.0,NaN,YES
1,Kisan Call Centre (KCC) - Transcripts of farme...,NaN,"agriculture, public service delivery, informat...",2024-07-12,2024-07-12,2025-06-27,Dataset,NaN,text/json,NaN,...,NaN,241962,219561,NaN,TRUE,1.0,6622307.0,1.0,NaN,1
2,All India pincode directory updated till last ...,All India Pincode Directory through Webservice...,"india pincodes, pincode directory, postal serv...",2020-04-12,2020-04-12,2025-06-27,Dataset,23760720,text/csv,https://www.indiapost.gov.in/vas/pages/findpin...,...,NaN,105072,177619,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Registrars of Companies (RoC)-wise Company Mas...,NaN,NaN,2015-09-15,2024-04-24,2025-06-27,Dataset,996505,text/csv,NaN,...,NaN,9035,39682,NaN,TRUE,1.0,603037492.0,1.0,1.0,1
4,Current Daily Price of Various Commodities fro...,NaN,NaN,2013-05-23,2013-05-23,2025-06-27,Dataset,112,text/csv,NaN,...,NaN,59612,37549,2,TRUE,1.0,3670701.0,1.0,1.0,NaN


In [11]:
#for filling in title, description, note wherever required.
# TODO Create a generic prompt template 
metadata_enhancement_prompt="""

# System Prompt: Metadata Enhancement for India Government Open Data

## Role
You are an expert metadata curator specializing in government open data standards including Dublin Core, DCAT v3, and India's data.gov.in specifications. Your task is to generate enhanced, standardized metadata values for dataset resources imported from India's data.gov.in platform, focusing on improving discoverability and clarity while maintaining accuracy.

## Instructions
Do not make mistakes. 

## Input Format
You will receive CSV rows with the following key columns:
- **dcterms:title**: Existing dataset title (primary source for enhancement)
- **dcterms:description**: Current description (may be empty or need improvement)
- **dcat:theme**: High-level thematic categories (semicolon-separated)
- **dcterms:spatial**: Geographic coverage (typically "India" or state names)
- **dcatin:jurisdictionLevel**: Government level (Central/State)
- **dct:accrualPeriodicity**: Update frequency (Daily, Monthly, Annual, etc.)
- **dcat:keyword**: Existing keywords (comma-separated)
- **dcat:contactPoint** related fields: Organization details
- **dcterms:publisher**: Publishing ministry/department
- **dcatin:note**: Additional contextual notes (often empty, needs generation)
- **dcat:temporalResolution**: Time granularity of data
- **dcterms:temporal**: Time period covered
- Other technical and administrative fields for context

## Task Requirements

### 1. Generate Enhanced Title (dcterms:title)
Create an improved title that:
- **Preserves core subject matter** from the original title
- **Adds clarity** by expanding abbreviations where appropriate
- **Includes temporal context** if relevant (e.g., "2021-2025" for time-series data)
- **Maintains consistency** in formatting and structure
- **Length**: 10-20 words typically, maximum 25 words
- **Format**: Title Case for major words
- **Avoids redundancy** with catalog-level information

**Enhancement Logic:**
- Start with the existing title as the base
- Expand known abbreviations (KCC → Kisan Call Centre, RoC → Registrars of Companies)
- Add geographic scope if not clear (append "in India" or state name when relevant)
- Include time period for historical datasets
- Ensure searchability by including key domain terms

### 2. Generate Comprehensive Description (dcterms:description)
Create or enhance description that:
- **Expands on the title** with detailed explanation (minimum 2-3 sentences)
- **Describes what the dataset contains** specifically
- **Explains the purpose and use cases** of the data
- **Mentions update frequency** and temporal coverage
- **Includes data structure details** (what fields/metrics are included)
- **References the source system** or collection methodology when known
- **Length**: 50-150 words typically

**Generation Logic:**
- If description exists: Enhance it by adding missing elements
- If description is empty: Generate from title, theme, keywords, and other metadata
- Include context about the publishing ministry/department
- Mention geographic and temporal scope explicitly
- Add information about data granularity (district-wise, state-wise, etc.)

### 3. Generate Contextual Note (dcatin:note)
Create informative note that:
- **Provides additional context** not covered in description
- **Explains data collection methodology** or source systems
- **Notes any special characteristics** or limitations
- **Includes update patterns** or processing information
- **Mentions related datasets** or services when applicable
- **Length**: 20-75 words typically

**Generation Logic:**
- Focus on operational and technical details
- Mention source portals or systems (e.g., "Generated through AGMARKNET Portal")
- Include processing or aggregation methods
- Note any data quality considerations
- Reference relevant policies (e.g., NDSAP compliance)
- Add sector-specific context

### 4. Generate Alternative Title (dcterms:alternative)
Create an alternative title that:
- Provides a different perspective or emphasis from the main title
- Uses simpler language or common terminology
- May include popular/colloquial terms used for the dataset
- Suitable for search engine optimization
- Length: 8-15 words typically
- Format: Title Case, may include parenthetical clarifications

### 5. Generate Short Description (dcterms:abstract)
Create a concise abstract that:
- Summarizes the dataset in 1-2 sentences maximum
- Focuses on the "what" and "why" of the data
- Suitable for quick previews and search results
- Avoids technical details
- Length: 20-40 words typically

### 6. Processing Rules

**For semicolon-separated values:**
- Parse and extract individual values
- Use for context but write in natural prose
- Don't replicate semicolon format in output

**For empty/null fields:**
- Generate appropriate content based on other available metadata
- Use publisher, theme, and keyword fields for context
- Infer from related fields when logical

**For temporal data:**
- Daily data → Emphasize real-time or near real-time nature
- Monthly/Quarterly → Highlight trending capabilities
- Annual → Focus on long-term analysis potential
- One-time → Note as snapshot or reference dataset

### 7. Quality Guidelines

**Title Quality Rules:**
1. Must be self-explanatory without requiring additional context
2. Include the primary subject, geographic scope, and time period where relevant
3. Avoid jargon unless widely recognized in the domain
4. Use consistent naming patterns for similar datasets

**Description Quality Rules:**
1. First sentence should summarize what the dataset is
2. Second sentence should explain what it contains
3. Additional sentences add context about source, frequency, and usage
4. Use active voice and present tense for current datasets
5. Include quantitative details where available (number of records, coverage)

**Note Quality Rules:**
1. Complement, don't duplicate the description
2. Focus on technical or operational details
3. Be concise but informative
4. Include actionable information for data users

## Output Format 
Return a valid JSON with the following structure:

{
  "enhanced_title": "Improved title text",
  "alternative_title": "Alternative title for better discoverability",
  "enhanced_description": "Comprehensive description text",
  "short_description": "Brief 1-2 sentence abstract",
  "generated_note": "Contextual note text",
  "processing_summary": {
    "title_changes": "Brief explanation of title improvements",
    "alternative_title_rationale": "Why this alternative was chosen",
    "description_changes": "What was added/enhanced in description",
    "short_description_rationale": "Short description representation rationale",
    "note_generation": "Basis for note content"
  },
  "metadata_quality_score": 0.85,
  "quality_score_explanation": "Brief explanation of how the score was calculated based on completeness and enhancement value",
  "source_fields_used": ["list", "of", "fields", "used"]
}


### Quality Score Calculation
The `metadata_quality_score` (0.0 to 1.0) should reflect:
- **Completeness** (40%): How many metadata fields were successfully enhanced
- **Enhancement Value** (30%): Degree of improvement over original metadata
- **Source Data Quality** (30%): Quality and completeness of input metadata

Scoring guidelines:
- 0.9-1.0: All fields enhanced with substantial improvements, rich source data
- 0.7-0.89: Most fields enhanced, moderate improvements, adequate source data
- 0.5-0.69: Basic enhancements, limited source data or minimal improvements
- Below 0.5: Insufficient source data or minimal enhancement possible

## Special Handling for Indian Government Data

### Ministry/Department Context:
- Recognize ministry hierarchies and use appropriate level of detail
- Include department names when they add specificity
- Use standard abbreviations (MoA, MoHFW, etc.) in notes but spell out in descriptions

### Geographic Indicators:
- "India" or "All India" for national datasets
- State names for state-level data
- "District-wise" or "State-wise" for granular data
- Include Union Territories when applicable

### Sector-Specific Terminology:
- Agriculture: Include terms like "Mandi", "Kharif", "Rabi" with explanations
- Finance: Reference financial years (FY 2023-24 format)
- Health: Include scheme names (NRHM, Ayushman Bharat)
- Education: Reference education levels (Primary, Secondary, Higher)

### Common Indian Government Acronyms to Expand:
- KCC → Kisan Call Centre
- PHC → Primary Health Centre
- MGNREGA → Mahatma Gandhi National Rural Employment Guarantee Act
- GSVA → Gross State Value Added
- MSP → Minimum Support Price
- FCI → Food Corporation of India
- NITI → National Institution for Transforming India

## Example Processing

**Input:**

dcterms:title: "Daily Wholesale Mandi Market Prices of Agricultural Commodities by Variety"
dcterms:description: "The data refers to prices of variety wise agricultural commodities..."
dcatin:note: [empty]
dcat:theme: "Agriculture, Agricultural Marketing"
dcatin:jurisdictionLevel: "Central"
dct:accrualPeriodicity: "Daily"


**Output:**

{
  "enhanced_title": "Daily Wholesale Market Prices of Agricultural Commodities by Variety across Indian Mandis",
  "alternative_title": "Mandi Prices for Farm Produce in India (Daily Updates)",
  "enhanced_description": "This dataset provides comprehensive daily wholesale price information for agricultural commodities differentiated by variety across regulated market yards (Mandis) in India. It includes maximum, minimum, and modal prices collected from Agricultural Produce Market Committees (APMCs) nationwide, enabling price discovery and market analysis. The data is updated daily through the AGMARKNET portal system, covering major food grains, pulses, oilseeds, spices, fruits, and vegetables. This information supports farmers in making informed selling decisions, helps traders identify arbitrage opportunities, and assists policymakers in monitoring food inflation and market dynamics.",
  "short_description": "Daily wholesale prices for agricultural commodities by variety from regulated mandis across India, updated through the AGMARKNET system.",
  "generated_note": "Data is sourced from over 3,000 regulated mandis through the AGMARKNET Portal (agmarknet.gov.in). Prices are reported by market officials and validated before publication. The dataset follows the Minimum Support Price (MSP) commodity classification system and is a critical input for agricultural policy formulation.",
  "processing_summary": {
    "title_changes": "Added 'across Indian Mandis' for geographic clarity and improved flow",
    "alternative_title_rationale": "Created simpler, search-friendly version using colloquial term 'Farm Produce' and emphasizing daily updates",
    "description_changes": "Expanded from 2 sentences to comprehensive 4-sentence description with use cases, source system, and beneficiary information",
    "short_description_rationale": "Distilled key information (what, where, how) into single sentence for quick scanning",
    "note_generation": "Created based on AGMARKNET system knowledge and agricultural market structure"
  },
  "metadata_quality_score": 0.90,
  "quality_score_explanation": "High score due to complete enhancement of all 5 fields with substantial value addition. Original metadata provided good foundation (title, partial description, theme). Successfully added geographic context, use cases, source system details, and created alternative title and abstract.",
  "source_fields_used": ["dcterms:title", "dcterms:description", "dcat:theme", "dcatin:jurisdictionLevel", "dct:accrualPeriodicity"]
}


## Error Handling
- If title is missing: Set quality score to 0.0 and return error in output
- If both description and note are empty: Generate from available metadata
- If minimal metadata available: Flag with low quality score (<0.5) and explain in quality_score_explanation
- For malformed inputs: Attempt best-effort processing with appropriate flags in processing_summary

## Constraints
- Maintain factual accuracy - don't invent information not implied by metadata
- Preserve all specific details from original title and description
- Ensure generated content is grammatically correct and professional
- Keep Indian English conventions (e.g., "Centre" not "Center")
- Respect character limits for practical display purposes


"""

In [6]:
# sector_df=pd.read_csv('/home/aakash/NIC/Newfolder/nic-metadata-cleaning/data/sector.csv')
# sector_df.head()
with open("/home/aakash/NIC/Newfolder/nic-metadata-cleaning/notebooks/sector.json", "r", encoding="utf-8") as f:
    sectors = json.load(f)

print(sectors)


{'Census and Surveys': ['Annual Health Survey', 'Census', 'Civil Registration System', 'National Population Register', 'Sample Registration System', 'Socio-economic & Caste Census'], 'Census': ['House listing and Housing Census', 'Population Enumerator'], 'Agriculture': ['Agricultural Marketing', 'Dairying', 'Agricultural Produces', 'Agricultural Research & Extension', 'Animal Husbandry', 'Crops', 'Fertilizers', 'Horticulture', 'Irrigation', 'Organic farming', 'Plant Protection', 'Seeds', 'Soil and Water Conservation', 'Fisheries', 'Sericulture'], 'Animal Husbandry': ['Fishery'], 'Art and Culture': ['Archaeology', 'Dance', 'Festivals', 'Handicrafts', 'Heritage', 'Literature', 'Monuments', 'Music', 'Painting', 'Theatre'], 'Commerce': ['Companies', 'Export', 'Import', 'SEZs', 'Trade promotion'], 'Parliament Of india': ['Rajya Sabha', 'Lok Sabha'], 'Water and Sanitation': ['Drinking Water', 'Sanitation'], 'Information and Communications': ['Information and Technology', 'Post', 'Telecom'],

In [14]:
# for keyword/theme generation
categorize_system_prompt='''
# System Prompt: Metadata Keyword Generation for Open Government Data

## Role
You are an expert metadata curator specializing in government open data standards including Dublin Core, DCAT v3, and data.gov.in specifications. Your task is to generate high-quality keywords and metadata classifications from dataset resource metadata. 
Some fields may contain semicolon separated values. Extract them individually to run processing. Also if similar semicolon separated values arise
use your best judgment to fix them and run downstream processing.

## Input Format
You will receive CSV rows with the following columns:
- title: Dataset resource title
- catalog_title: Catalog-level description
- sector: Top-level sector tags (semicolon-separated)
- sector_resource: Resource-level granular sector tags (semicolon-separated, may be null)
- ministry_department: Governing ministry/department (semicolon-separated)
- state_department: State-level department (may be null)
- note: Additional notes about the dataset (may be null)
- frequency: Update frequency (e.g., Daily, Monthly)
- granularity: Data granularity level
- govt_type: Government level (Central/State)
- Other technical fields for context

## Task Requirements

### **1. Generate Enhanced Layman Keywords (`dcat:keyword`)**

Produce **4–10 layman-friendly keywords** that:

* Reflect the **core subject** of the dataset
* Use **simple, everyday language** understood by non-experts
* Capture both **broad and specific** dataset aspects (without jargon)
* Are helpful for discovery through general search (data.gov.in, Google, etc.)

**Keyword Rules**

* Each keyword must be **1–2 words only**
* Use **lowercase**; no punctuation except a single space for two-word terms
* Avoid duplicates
* Prefer only  **singular forms**, and not plural form. 
* Exclude technical or internal terms (e.g., “api”, “metadata”, “hmis”, “dcat”)
* Ignore ministry/department names and acronyms
* Parse semicolon-separated values into individual hints
* Use **India-relevant terms** when suitable (e.g., “mandi”, “rainfall”, “budget”)

---

### **2. Generate Sponsored Keywords**

Sponsored keywords represent **the most relevant, high-similarity search terms** directly tied to the dataset’s **title** or **note**.
These should:

* Be **1–2 words long**, short and intuitive
* Represent **the dataset’s main idea** or purpose
* Not include department names, locations (unless inherent), or administrative words
* Be a **subset or close match** of what a user might naturally type when searching for the dataset title

You should generate **3–5 sponsored keywords** that are:

* Derived from the **strongest overlaps** between the title/note and layman terms
* Ranked by relevance (first is most relevant)
* Still follow all the layman keyword rules (lowercase, no punctuation, ≤2 words)


**Extraction Logic:**
- Parse semicolon-separated values in sector, sector_resource, ministry_department
- Use your best judgment to create keywords from the existing metadata fields.
- You may come up with assumptions and best case tagging as well.
- Some fields may contain semicolon separated values. Extract them individually to run processing.


### 3. Generate Theme 
Identify high-level thematic categories that:
- Represent broad policy/domain areas
- Align with government data categorization schemes
- Enable cross-dataset discovery
- Map to standard taxonomies (e.g., COFOG - Classification of Functions of Government)
- Enable cross-dataset discovery

You should only refer to the following controlled vocabulary 'SECTOR_VOCAB_JSON' for subject classification which has sector for eg. "Census and Surveys" and its sub-sectors for e.g. "Annual Health Survey". If a suitable subject cannot be found, do not invent new subjects. 
If there is no sub sector that can be used to represent the dataset, use the high level or sectors to map to the subject. 
The generated themes will be used for replacing the current Resource-level granular sector tags. 

SECTOR_VOCAB_JSON:
{
  "Census and Surveys": [
    "Annual Health Survey",
    "Census",
    "Civil Registration System",
    "National Population Register",
    "Sample Registration System",
    "Socio-economic & Caste Census"
  ],
  "Census": [
    "House listing and Housing Census",
    "Population Enumerator"
  ],
  "Agriculture": [
    "Agricultural Marketing",
    "Dairying",
    "Agricultural Produces",
    "Agricultural Research & Extension",
    "Animal Husbandry",
    "Crops",
    "Fertilizers",
    "Horticulture",
    "Irrigation",
    "Organic farming",
    "Plant Protection",
    "Seeds",
    "Soil and Water Conservation",
    "Fisheries",
    "Sericulture"
  ],
  "Animal Husbandry": [
    "Fishery"
  ],
  "Art and Culture": [
    "Archaeology",
    "Dance",
    "Festivals",
    "Handicrafts",
    "Heritage",
    "Literature",
    "Monuments",
    "Music",
    "Painting",
    "Theatre"
  ],
  "Commerce": [
    "Companies",
    "Export",
    "Import",
    "SEZs",
    "Trade promotion"
  ],
  "Parliament Of india": [
    "Rajya Sabha",
    "Lok Sabha"
  ],
  "Water and Sanitation": [
    "Drinking Water",
    "Sanitation"
  ],
  "Information and Communications": [
    "Information and Technology",
    "Post",
    "Telecom"
  ],
  "Defence": [
    "Air Force",
    "Army",
    "Navy",
    "Para Military Forces"
  ],
  "Economy": [
    "Prices",
    "Macro Economy"
  ],
  "Education": [
    "Adult Education",
    "Elementary",
    "Higher Education",
    "Secondary"
  ],
  "Environment and Forest": [
    "Bio-diversity",
    "Biomedical Waste",
    "Ecology",
    "Forest",
    "Forest Resources",
    "Hazardous Waste",
    "Industrial Air Pollution",
    "Municipal Waste",
    "Noise Pollution",
    "Residential Air Pollution",
    "Vehicular Air Pollution",
    "Water Quality",
    "Natural Resources",
    "Sanitation",
    "Wild life"
  ],
  "Water Resources": [
    "Drinking Water",
    "Ground Water",
    "Surface Water"
  ],
  "Finance": [
    "Banking",
    "Economy",
    "Revenue",
    "Insurance",
    "Pension Reforms"
  ],
  "Food": [
    "Consumer Affairs",
    "Consumer Cooperatives",
    "Public Distribution"
  ],
  "Foreign Affairs": [
    "Consulates",
    "Embassy",
    "NRI",
    "Passport",
    "Visa"
  ],
  "Governance and Administration": [
    "Constitution",
    "District Adminstration",
    "Grievances",
    "Local Government",
    "Lok Sabha",
    "Pensions",
    "Rajya Sabha",
    "State Legislative",
    "Union/State Government Administration"
  ],
  "Health and Family welfare": [
    "Family Welfare",
    "Health"
  ],
  "Home Affairs and Enforcement": [
    "Enforcement Organizations",
    "Internal Security",
    "Police"
  ],
  "Housing": [
    "EWS Housing",
    "Rural Housing",
    "Urban Housing"
  ],
  "Industries": [
    "Chemicals and Petrochemicals",
    "Corportae governance",
    "Cottage",
    "Defence Products",
    "Food Processing",
    "Heavy",
    "Insurance",
    "Manufacturing",
    "Medium",
    "Micro",
    "Petroleum and Natural Gas",
    "Pharmaceuticals",
    "Retail",
    "Small Scale",
    "Textiles",
    "Tourism"
  ],
  "Textiles": [
    "Sericulture"
  ],
  "Information and Broadcasting": [
    "Broadcasting",
    "Film",
    "Print Media"
  ],
  "Infrastructure": [
    "Bridges",
    "Dams",
    "Power",
    "Roads"
  ],
  "Urban": [
    "Development"
  ],
  "Judiciary": [
    "District Court",
    "High Court",
    "Subordinate Court",
    "Supreme Court"
  ],
  "Labour and Employment": [
    "Employment",
    "Organized Sector Workers",
    "Unorganized Sector Workers"
  ],
  "Power and Energy": [
    "Non Renewable",
    "Renewable"
  ],
  "Rural": [
    "Development",
    "Land Resources",
    "Panchayati Raj"
  ],
  "Science and Technology": [
    "Atmospheric Science",
    "Coastal & Island",
    "Earth Sciences",
    "Geo Technology",
    "Marine Science",
    "Polar Science",
    "Research & Development"
  ],
  "Social Development": [
    "Children",
    "Disabled",
    "Minority",
    "Tribal",
    "Women"
  ],
  "Transport": [
    "Aviation",
    "Metro",
    "Railways",
    "Road Transport",
    "Water ways"
  ],
  "Travel and Tourism": [
    "Lodging",
    "Modes of Travel",
    "Places"
  ],
  "Youth and Sports": [
    "Games",
    "Youth Affairs"
  ]
}

output example: 
Information and Communications; Post


### 4. Provide Justification
For each generated field, explain:
- Which source columns were used
- What extraction/normalization rules were applied
- Why specific terms were selected or excluded
- Any ambiguities or assumptions made

### 5. Confidence Scoring
Assign confidence scores (0.0 to 1.0) based on:
- **1.0**: All source fields present with clear, unambiguous information
- **0.9**: Minor ambiguity or one secondary field missing
- **0.8**: Some interpretation required, multiple valid classifications possible
- **0.7**: Significant missing information (e.g., note, sector_resource null)
- **0.6**: Sparse metadata, heavy inference required
- **<0.6**: Insufficient metadata for reliable classification

## Output Format
Return a valid JSON with the following additional keys:
- generated_keywords: Comma-separated list of keywords
- generated_sponsored_keywords: Comma-separated list of sponsored keywords
- generated_theme: Comma-separated list of themes
- justification: Detailed reasoning for selections
- confidence_score: Float value 0.0-1.0
- metadata_gaps: Fields that were null/missing affecting quality


## Quality Guidelines

### Keyword Quality Rules:
1. Avoid redundancy with title terms unless critical
2. Include Hindi/regional language terms where applicable to Indian context
3. Use lowercase unless proper nouns
4. Separate compound concepts (e.g., "price data" → "prices", "market data")
5. Use your best judgment to structure the output JSON shape. 


### Theme Quality Rules:
1. Use the list of sectors and sub-sectors in sector_vocab only
2. Limit to 2-4 to maintain meaningful categorization
3. if Sub sector is being used it should always have its parent sector included.
4. Sub sectors and sectors should be separated by ';' and if there are more than one sector and sub sector they should be added by using ",". For example: "Agriculture; Agricultural Marketing, Health and Family Welfare; Health" 
5. Consider cross-cutting themes (e.g., "Public Services", "Economic Development")

## Special Handling

### Ministry/Department Processing:
- Extract parent ministry from hierarchical strings
- Use abbreviated forms if widely recognized (e.g., MoA for Ministry of Agriculture)
- Include as contextual keywords only if domain-relevant

### Null Field Handling:
- If sector_resource is null, rely heavily on sector and title
- If note is null, reduce confidence score by 0.1
- If state_department is null and govt_type is "Central", this is expected

### Frequency/Granularity Integration:
- Include temporal keywords for real-time/daily data (e.g., "daily", "real-time monitoring")
- Add "historical" for archived datasets
- Include "time-series" for temporally granular data

## Example Processing Logic

**Input:**
```
title: "Variety-wise Daily Market Prices Data of Commodity"
sector: "Agriculture;Agricultural Marketing"
ministry_department: "Ministry of Agriculture and Farmers Welfare;Department of Agriculture..."
frequency: "Daily"
```

**Processing Steps:**
1. Parse sector → ["Agriculture", "Agricultural Marketing"]
2. Extract from title → "variety", "market prices", "commodity"
3. Identify domain term → "mandi" (implied from catalog_title context)
4. Add temporal → "daily prices"
5. Add format → "price data"

**Output:**
- Keywords: agricultural commodities, market prices, mandi prices, daily data, crop varieties, agricultural marketing, price monitoring
- Subject: Agricultural Economics, Market Information Systems, Commodity Trading
- Theme: Agriculture, Economic Development
- Confidence: 0.85 (sector_resource null, but strong other fields)

## Constraints
- Maximum 10 keywords per resource
- Maximum 5 subjects per resource
- Maximum 4 themes per resource
- All outputs in English (add transliterated terms where culturally relevant)
- Maintain data.gov.in vocabulary consistency where applicable

## Error Handling
If critical fields (title, sector) are missing or malformed:
- Set confidence_score to 0.3
- Flag in metadata_gaps column
- Provide best-effort keywords from available fields
- Add justification note: "INSUFFICIENT_METADATA"
- If a column contains semicolon (;) separated values, first extract each value and then run processing.

'''

In [ ]:

# for keyword generation
keyword_prompt= '''


# **System Prompt: Metadata Keyword Generation for Open Government Data (Layman + Sponsored Keywords)**

## **Role**

You are an expert metadata curator specializing in government open data standards (Dublin Core, DCAT v3, data.gov.in).
Your task is to generate **layman-friendly search keywords** and **sponsored keywords** from dataset metadata.
The output should be concise, human-readable, and optimized for search discoverability by non-expert users.



## **Input Format**

You will receive CSV rows with the following columns (some may be empty or semicolon-separated):

* **title**: Dataset or resource title
* **catalog_title**: Catalog-level description
* **sector**: Broad sector tags (semicolon-separated)
* **sector_resource**: Granular sector tags (semicolon-separated, may be null)
* **ministry_department**: Governing ministry or department (semicolon-separated)
* **state_department**: State-level department (may be null)
* **note**: Additional notes or descriptions (may be null)
* **frequency**: Update frequency (e.g., Daily, Monthly)
* **granularity**: Level of data detail
* **govt_type**: Government level (Central or State)
* Other technical fields for context

---

## **Task Requirements**

### **1. Generate Enhanced Layman Keywords (`dcat:keyword`)**

Produce **6–10 layman-friendly keywords** that:

* Reflect the **core subject** of the dataset
* Use **simple, everyday language** understood by non-experts
* Capture both **broad and specific** dataset aspects (without jargon)
* Are helpful for discovery through general search (data.gov.in, Google, etc.)

**Keyword Rules**

* Each keyword must be **1–2 words only**
* Use **lowercase**; no punctuation except a single space for two-word terms
* Avoid duplicates
* Prefer only  **singular forms**, and not plural form. 
* Exclude technical or internal terms (e.g., “api”, “metadata”, “hmis”, “dcat”)
* Ignore ministry/department names and acronyms
* Parse semicolon-separated values into individual hints
* Use **India-relevant terms** when suitable (e.g., “mandi”, “rainfall”, “budget”)

---

### **2. Generate Sponsored Keywords**

Sponsored keywords represent **the most relevant, high-similarity search terms** directly tied to the dataset’s **title** or **note**.
These should:

* Be **1–2 words long**, short and intuitive
* Represent **the dataset’s main idea** or purpose
* Not include department names, locations (unless inherent), or administrative words
* Be a **subset or close match** of what a user might naturally type when searching for the dataset title

You should generate **3–5 sponsored keywords** that are:

* Derived from the **strongest overlaps** between the title/note and layman terms
* Ranked by relevance (first is most relevant)
* Still follow all the layman keyword rules (lowercase, no punctuation, ≤2 words)

---

### **3. Output Format**

Return **exactly three lines** (and nothing else).
The output must be valid JSON with the following format:

```
Title: <copy the title exactly as provided>
Enhanced Keywords: <kw1, kw2, kw3, kw4, kw5, kw6, kw7, kw8>
Sponsored Keywords: <kw1, kw2, kw3, kw4>
```

**Do not include:**

* Explanations, reasoning, or scores
* Department or ministry names
* Special symbols, punctuation, or formatting beyond this structure

---

## **Quality Guidelines**

* Focus on **discoverability** by laypeople, not metadata experts
* Keywords should make sense when read aloud (“someone would search this”)
* Avoid redundancy between Enhanced and Sponsored lists
* If `sector_resource` is missing, rely on title, catalog_title, and note
* Sponsored keywords should always sound like **real-world search queries** (e.g., “market prices”, “rainfall data”, “railway schedule”)

---

## **Error Handling**

* If the `title` field is empty, write: `Title: ` (keep it blank after the colon).
* Still generate both **Enhanced Keywords** and **Sponsored Keywords** to the best of your ability.
* Always return **exactly three lines** with no extra text, comments, or formatting.

---

## **Example**

**Input:**

```
title: "Variety-wise Daily Market Prices Data of Commodity"
sector: "Agriculture;Agricultural Marketing"
ministry_department: "Ministry of Agriculture and Farmers Welfare"
frequency: "Daily"
note: "Contains mandi price information for various crops and commodities."
```

**Output (in JSON):**

```
Title: Variety-wise Daily Market Prices Data of Commodity
Enhanced Keywords: mandi, market prices, crop prices, wholesale, daily data, agriculture, mandi rates, price trends, commodity prices
Sponsored Keywords: market prices, mandi prices, crop prices, commodity prices
```

---
# for HVD classification
You will receive an additional field in the input:

* `HVD Flag:` either `0` or `1`.

Interpret it as:

* `0` → the dataset is **not** marked as High Value.
* `1` → the dataset **is** marked as High Value and must be classified into one of the following categories:

1. `geospatial`
2. `earth observation and environment`
3. `meteorological`
4. `statistics`
5. `companies and company ownership`
6. `mobility`

#### HVD Rules

* If `HVD Flag` is `0`:
  * Do **not** classify it into any category.
  * Set `"hvd_category"` to an empty string `""`.

* If `HVD Flag` is `1`:
  * Use the dataset **Title**, **Catalog Title**, and **Note/Description** to decide the **single best-fit** category from the list above.
  * Output the category name exactly as one of:
    * `"geospatial"`, `"earth observation and environment"`, `"meteorological"`, `"statistics"`, `"companies and company ownership"`, `"mobility"`.


'''

In [15]:
def get_keywords(metadata_content):
    response=client.chat.completions.create(
         model='gpt-5-nano',
         temperature=1,
         response_format={
             "type":"json_object",
        
         },
         messages=[
              {
                  "role":"system",
                  "content":categorize_system_prompt
              },
              {
                  "role":"user",
                  "content":metadata_content
              }
         ],

    )
    return json.loads(response.choices[0].message.content)

In [ ]:
def get_enhanced_metadata(metadata_content):
    response = client.chat.completions.create(
        model='gpt-5-nano',
        temperature=1,
        response_format={"type": "json_object"},
        messages=[
            {
                "role": "system",
                "content": categorize_system_prompt
            },
            {
                "role": "user",
                "content": metadata_content
            }
        ],
    )
    return json.loads(response.choices[0].message.content)

In [9]:
df=pd.read_csv("/home/aakash/NIC/cdl-updated/nic-metadata-cleaning/data/ogd_metadata_sample/sample_dataset_nic.csv", encoding="cp1252", )
df.head()

,title,resource_category,description,catalog_title,govt_type,ministry_department,state_department,published_date,changed,created,...,field_high_value_dataset,field_show_export,cdos_state_ministry,is_rated,external_api_reference,note,node_alias,ogdp_download_count,ogdp_view_count,domain
0,Variety-wise Daily Market Prices Data of Commo...,Dataset,The data refers to prices of variety wise agri...,Current daily price of various commodities fro...,Central,Ministry of Agriculture and Farmers Welfare;De...,NaN,02-06-2024,27-06-2025,21-05-2024,...,1,True,Directorate of Marketing and Inspection (DMI),1,6622308.0,NaN,/resource/variety-wise-daily-market-prices-dat...,394032,167942,data.gov.in
1,Kisan Call Centre (KCC) - Transcripts of farme...,Dataset,This dataset comprises transcripts of farmers'...,District wise and month wise queries of farmer...,Central,Ministry of Agriculture and Farmers Welfare;De...,NaN,12-07-2024,27-06-2025,12-07-2024,...,1,True,Department of Agriculture and Farmers Welfare,1,6622307.0,NaN,/resource/kisan-call-centre-kcc-transcripts-fa...,219561,241962,data.gov.in
2,All India Pincode Directory till last month,Dataset,All India Pincode Directory through Webservice...,All India Pincode Directory (Through WebService),Central,Ministry of Communications;Department of Posts,NaN,04-12-2020,27-06-2025,04-12-2020,...,0,True,Department of Posts,1,6818292.0,Data will be updated on monthly basis.,/resource/all-india-pincode-directory-till-las...,105072,177619,data.gov.in
3,Registrars of Companies (RoC)-wise Company Mas...,Dataset,This dataset provides monthly RoC-wise master ...,Company Master Data,Central,Ministry of Corporate Affairs,NaN,24-04-2024,27-06-2025,15-09-2015,...,1,True,Ministry of Corporate Affairs,1,603037492.0,Figures Authorized Capital and Paid up Capital...,/resource/registrars-companies-roc-wise-compan...,39682,9035,data.gov.in
4,Current Daily Price of Various Commodities fro...,Dataset,This dataset records daily wholesale price obs...,Current daily price of various commodities fro...,Central,Ministry of Agriculture and Farmers Welfare;De...,NaN,23-05-2013,27-06-2025,23-05-2013,...,1,True,Directorate of Marketing and Inspection (DMI),1,3670701.0,NaN,/resource/current-daily-price-various-commodit...,37549,59612,data.gov.in


In [10]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import csv
from threading import Lock

In [11]:
def extract_clean_value(value):
    """Clean and extract value from field"""
    if pd.isna(value) or value == 'nan' or value == '':
        return ""
    return str(value).strip()

def parse_semicolon_separated(value):
    """Parse semicolon-separated values into a list"""
    if not value or pd.isna(value):
        return []
    return [v.strip() for v in str(value).split(';') if v.strip()]

In [ ]:
def process_row_generation(row):
    """Process a single row"""
    # Extract all relevant fields from the actual CSV structure
    # Primary fields for enhancement
    title = extract_clean_value(row.get("dcterms:title", ""))
    description = extract_clean_value(row.get("dcterms:description", ""))
    existing_note = extract_clean_value(row.get("dcatin:note", ""))
    existing_alternative = extract_clean_value(row.get("dcterms:alternative", ""))
    existing_abstract = extract_clean_value(row.get("dcterms:abstract", ""))
    
    # Supporting metadata fields
    themes = parse_semicolon_separated(row.get("dcat:theme", ""))
    spatial = extract_clean_value(row.get("dcterms:spatial", ""))  # Removed space
    temporal = extract_clean_value(row.get("dcterms:temporal", ""))
    jurisdiction = extract_clean_value(row.get("dcatin:jurisdictionLevel", ""))
    frequency = extract_clean_value(row.get("dct:accrualPeriodicity", ""))
    temporal_res = extract_clean_value(row.get("dcat:temporalResolution", ""))
    keywords = extract_clean_value(row.get("dcat:keyword", ""))
    publisher = extract_clean_value(row.get("dcterms:publisher", ""))
    creator = extract_clean_value(row.get("dcterms:creator", ""))
    contact_name = extract_clean_value(row.get("vCard:fn", ""))  # Removed spaces
    org_name = extract_clean_value(row.get("vcard:organization-name", ""))  # Removed spaces
    data_type = extract_clean_value(row.get("dcterms:type", ""))
    media_type = extract_clean_value(row.get("dcat:mediaType", ""))
     
    # Build metadata content string for the LLM
    metadata_content = f'''
=== CURRENT METADATA ===
Title: {title}
Description: {description}
Existing Note: {existing_note}
Existing Alternative Title: {existing_alternative}
Existing Abstract: {existing_abstract}

=== CONTEXTUAL INFORMATION ===
Themes: {'; '.join(themes) if themes else 'Not specified'}
Keywords: {keywords if keywords else 'Not specified'}
Publisher: {publisher if publisher else 'Not specified'}
Creator: {creator if creator else 'Not specified'}
Organization: {org_name if org_name else 'Not specified'}
Contact Person: {contact_name if contact_name else 'Not specified'}

=== COVERAGE DETAILS ===
Spatial Coverage: {spatial if spatial else 'India'}
Temporal Coverage: {temporal if temporal else 'Not specified'}
Jurisdiction Level: {jurisdiction if jurisdiction else 'Not specified'}

=== DATA CHARACTERISTICS ===
Update Frequency: {frequency if frequency else 'Not specified'}
Temporal Resolution: {temporal_res if temporal_res else 'Not specified'}
Data Type: {data_type if data_type else 'Dataset'}
Media Type: {media_type if media_type else 'Not specified'}

TASK: Generate enhanced metadata values for all required fields according to the system prompt.
'''
    
    try:
        result = get_enhanced_metadata(metadata_content)
        logging.info(f"Processed: {title[:50]}...")
        
    except Exception as e:
        logging.error(f"LLM failed for '{title}': {e}")
        result = {
            "enhanced_title": title,
            "alternative_title": "",
            "enhanced_description": description,
            "short_description": "",
            "generated_note": "",
            "processing_summary": {
                "title_changes": "",
                "alternative_title_rationale": "",
                "description_changes": "",
                "short_description_rationale": "",
                "note_generation": "",
                "error": str(e)
            },
            "metadata_quality_score": 0.0,
            "quality_score_explanation": f"Processing failed: {str(e)}",
            "source_fields_used": []
        }
    
    return {
        "original_title": str(title),
        "original_description": str(description)[:500],
        "original_note": str(existing_note),
        "original_alternative": str(existing_alternative),
        "original_abstract": str(existing_abstract),
        "enhanced_title": result.get("enhanced_title", title),
        "alternative_title": result.get("alternative_title", ""),
        "enhanced_description": result.get("enhanced_description", description),
        "short_description": result.get("short_description", ""),
        "generated_note": result.get("generated_note", ""),
        "processing_summary": result.get("processing_summary", {}),
        "quality_score": result.get("metadata_quality_score", 0.0),
        "quality_score_explanation": result.get("quality_score_explanation", ""),
        "source_fields_used": result.get("source_fields_used", []),
        "themes": str('; '.join(themes) if themes else ''),
        "publisher": str(publisher),
        "jurisdiction": str(jurisdiction),
        "frequency": str(frequency),
        "llm_response": json.dumps(result)
    }

In [ ]:
# Main execution
input_file = "/home/prajna/civicdatalab/nic-metadata/nic-metadata-cleaning/data/original/metadata_transformation.csv"
output_file = "/home/prajna/civicdatalab/nic-metadata/nic-metadata-cleaning/data/metadata_cleaned_100_alt_title_short_description.csv"
df = pd.read_csv(input_file)
print(f"Loaded {len(df)} rows")
csv_lock = Lock()

fieldnames = [
    # Original fields
    "original_title", 
    "original_description", 
    "original_note",
    "original_alternative", 
    "original_abstract",
    
    # Enhanced fields
    "enhanced_title",
    "alternative_title", 
    "enhanced_description",
    "short_description",
    "generated_note",
    
    # Context fields
    "themes", 
    "keywords", 
    "publisher", 
    "creator", 
    "organization",
    "jurisdiction", 
    "frequency", 
    "spatial", 
    "temporal",
    
    # Processing metadata
    "quality_score",
    "quality_score_explanation",
    "title_changes",
    "alternative_title_rationale",
    "description_changes",
    "short_description_rationale",
    "note_generation",
    "source_fields_used",
    "llm_response"
]

with open(output_file, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()

results_array = []
max_workers = 8
rows_to_process = df.head(105)  # Change to df for all rows

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    future_to_row = {
        executor.submit(process_row_generation, row): idx 
        for idx, row in rows_to_process.iterrows()
    }
    
    for future in as_completed(future_to_row):
        try:
            result_obj = future.result()
            
            # Flatten processing_summary into individual fields
            processing_summary = result_obj.get("processing_summary", {})
            csv_row = {
                # Original fields
                "original_title": result_obj.get("original_title", ""),
                "original_description": result_obj.get("original_description", ""),
                "original_note": result_obj.get("original_note", ""),
                "original_alternative": result_obj.get("original_alternative", ""),
                "original_abstract": result_obj.get("original_abstract", ""),
                
                # Enhanced fields
                "enhanced_title": result_obj.get("enhanced_title", ""),
                "alternative_title": result_obj.get("alternative_title", ""),
                "enhanced_description": result_obj.get("enhanced_description", ""),
                "short_description": result_obj.get("short_description", ""),
                "generated_note": result_obj.get("generated_note", ""),
                
                # Context fields
                "themes": result_obj.get("themes", ""),
                "keywords": result_obj.get("keywords", ""),
                "publisher": result_obj.get("publisher", ""),
                "creator": result_obj.get("creator", ""),
                "organization": result_obj.get("organization", ""),
                "jurisdiction": result_obj.get("jurisdiction", ""),
                "frequency": result_obj.get("frequency", ""),
                "spatial": result_obj.get("spatial", ""),
                "temporal": result_obj.get("temporal", ""),
                
                # Processing metadata
                "quality_score": result_obj.get("quality_score", 0.0),
                "quality_score_explanation": result_obj.get("quality_score_explanation", ""),
                "title_changes": processing_summary.get("title_changes", ""),
                "alternative_title_rationale": processing_summary.get("alternative_title_rationale", ""),
                "description_changes": processing_summary.get("description_changes", ""),
                "short_description_rationale": processing_summary.get("short_description_rationale", ""),
                "note_generation": processing_summary.get("note_generation", ""),
                "source_fields_used": json.dumps(result_obj.get("source_fields_used", [])),
                "llm_response": result_obj.get("llm_response", "")
            }
            
            results_array.append(csv_row)
            
            with csv_lock:
                with open(output_file, 'a', newline='', encoding='utf-8') as f:
                    writer = csv.DictWriter(f, fieldnames=fieldnames)
                    writer.writerow(csv_row)
            
            print(f"✓ {csv_row['original_title'][:60]}...")
            
        except Exception as e:
            logging.error(f"Row failed: {e}")

print(f"\n✓ Complete! Results saved to {output_file}")

Loaded 100 rows


2025-10-29 12:32:07,253 : INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-10-29 12:32:07,266 : INFO - Processed: Registrars of Companies (RoC)-wise Company Master ...
2025-10-29 12:32:07,272 : ERROR - Row failed: dict contains fields not in fieldnames: 'enhanced_description', 'generated_note', 'original_note', 'enhanced_title'
2025-10-29 12:32:08,089 : INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-10-29 12:32:08,093 : INFO - Processed: Current Daily Price of Various Commodities from Va...
2025-10-29 12:32:08,098 : ERROR - Row failed: dict contains fields not in fieldnames: 'enhanced_description', 'generated_note', 'original_note', 'enhanced_title'
2025-10-29 12:32:08,227 : INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-10-29 12:32:08,234 : INFO - Processed: Kisan Call Centre (KCC) - Transcripts of farmers q...
2025-10-29 12:32:08,237 : ERROR - Row faile


✓ Complete! Results saved to /home/prajna/civicdatalab/nic-metadata/nic-metadata-cleaning/data/metadata_cleaned_100_alt_title_short_description.csv


In [12]:
import csv
from threading import Lock
from concurrent.futures import ThreadPoolExecutor, as_completed 
def process_row_keyword(row):
    """Process a single row - thread-safe"""
    title = row.get("title", "")
    catalog_title = row.get("catalog_title", "")
    ministry_department = row.get("ministry_department", "")
    sector = row.get("sector", "")
    sector_resource = row.get("sector_resource", "")
    cdos_state_ministry = row.get("cdos_state_ministry", "")
    note = row.get("note", "")
   
    metadata_content = f'''
    Title: {title}
    Catalog Title: {catalog_title}
    Ministry/Department: {ministry_department}
    Sector: {sector}
    Sector Resource: {sector_resource}
    CDOS State Ministry: {cdos_state_ministry}
    Note: {note}
    '''
   
    result = get_keywords(metadata_content) #JSON
    logging.info(f"Result is {result}")
   
    return {
        "title": title,
        "sector": sector,
        "metadata_input": metadata_content,
        "llm_response": json.dumps(result),  # JSON --> JSON String for CSV
        "generated_keywords": result.get("generated_keywords", ""),
        "generated_sponsored_keywords": result.get("generated_sponsored_keywords", ""),
        "generated_theme": result.get("generated_theme", ""),
        "justification": result.get("justification", ""),
        "confidence_score": result.get("confidence_score", 0),
        "metadata_gaps": json.dumps(result.get("metadata_gaps", []))  # List to JSON string
    }

# Setup CSV file
output_file = "results_100.csv"
csv_lock = Lock()  # Thread-safe file writes

# Write header
fieldnames = [
    "title", "sector", "metadata_input", "llm_response",
    "generated_keywords", "generated_sponsored_keywords", "generated_theme",
    "justification", "confidence_score", "metadata_gaps"
]

with open(output_file, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()

# Parallel processing
results_array = []
max_workers = 8

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    future_to_row = {
        executor.submit(process_row_keyword, row): idx 
        for idx, row in df[:102].iterrows()
    }
    
    for future in as_completed(future_to_row):
        try:
            result_obj = future.result()
            results_array.append(result_obj)
            
            # Thread-safe incremental write
            with csv_lock:
                with open(output_file, 'a', newline='', encoding='utf-8') as f:
                    writer = csv.DictWriter(f, fieldnames=fieldnames)
                    writer.writerow(result_obj)
            
            print(f"TITLE: {result_obj['title']}\nSECTOR: {result_obj['sector']}\n\n✓ Written to CSV")
            print("\n----------------------------\n")
            
        except Exception as e:
            logging.error(f"Row processing failed: {e}")

print(f"\n✓ Complete! Results saved to {output_file}")

2026-01-15 10:35:21,721 : INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-15 10:35:21,743 : INFO - Result is {'generated_keywords': 'health data, monthly data, sub district, health indicator, public health, health statistics, district level', 'generated_sponsored_keywords': 'health data, monthly indicator, sub district, health facility, district data', 'generated_theme': 'Health, Family Welfare', 'justification': "Source fields used: title (for temporal and scope clues), sector (Health and Family welfare; Health; Family Welfare), sector_resource (repeated health-related terms to reinforce domain), note (data origin by health facilities across states/UTs), ministry_department (to be ignored for keywording per rules). Extraction/normalization rules applied: parsed semicolon-separated values in sector and sector_resource to assemble domain concepts; avoided acronyms and department names in keywords; ensured 1–2 word layman terms and used sing

TITLE: Health indicator-wise monthly datasets at sub district level from HMIS
SECTOR: Health and Family welfare;Family Welfare;Health

✓ Written to CSV

----------------------------



2026-01-15 10:35:23,619 : INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-15 10:35:23,622 : INFO - Result is {'generated_keywords': 'air quality, real time, air monitoring, air pollution, environment, location, live data', 'generated_sponsored_keywords': 'real time, air quality, air monitoring, live data', 'generated_theme': 'Industrial Air Pollution, Residential Air Pollution, Vehicular Air Pollution', 'justification': 'Source columns used: title (Real time Air Quality Index from various locations) to identify core concepts; sector and sector_resource (Environment and Forest; Industrial Air Pollution; Residential Air Pollution; Vehicular Air Pollution) to map to SECTOR_VOCAB_JSON sub-sectors; note (mentions real-time live data and potential data quality issues) to capture timeliness and data quality context; ministerial fields were ignored for keywords per guidelines. Extraction rules: split semicolon-separated values in sector/sector_res

TITLE: Real time Air Quality Index from various locations
SECTOR: Environment and Forest;Industrial Air Pollution;Residential Air Pollution;Vehicular Air Pollution

✓ Written to CSV

----------------------------



2026-01-15 10:35:25,464 : INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-15 10:35:25,468 : INFO - Result is {'generated_keywords': 'company data, registrar data, company records, master data, rupee capital, paid capital, authorized capital, company details', 'generated_sponsored_keywords': 'registrar data, company data, master data, company records', 'generated_theme': 'Commerce, Companies', 'justification': 'Source extraction: used title and sector fields. Sector was semicolon-separated into two hints: Commerce and Companies, which map to SECTOR_VOCAB_JSON under the Commerce theme and its sub-sector Companies. The note provides capital-related context (capital values in INR), which informed inclusion of capital-related keywords and currency term. Keywords were constrained to 1–2 words, lowercase, singular where possible, and avoided acronyms or department names. From the title “Registrars of Companies (RoC)-wise Company Master Data,” the

...

TITLE: State-wise Details of the Infant Mortality Rate, Institutional Delivery and Prevalence of Under Weight Children under five years of Age (in Reply to Unstarred Question on 07 December, 2022)
SECTOR: All

✓ Written to CSV

----------------------------



2026-01-15 10:44:58,294 : INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-15 10:44:58,296 : INFO - Result is {'generated_keywords': 'reservoir, water resource, ground water, surface water, water boundary, boundary, map data, water project', 'generated_sponsored_keywords': 'reservoir, boundary, water resource, map data, water project', 'generated_theme': 'Water Resources, Infrastructure', 'justification': 'Source columns used: title, catalog_title, sector, sector_resource, note (and to a lesser extent ministry_department for context). Extraction rules applied: 1) Split semicolon-separated fields (sector and sector_resource) into individual tokens and normalize to singular, 1–2 word layman terms (e.g., ground water, surface water). 2) Derive core concepts from the title and catalog_title (reservoir, water resources, boundaries) and confirm with the note (mentions reservoirs, entire country, information in multiple formats). 3) Generate 4–10 

TITLE: Shape file of reservoir
SECTOR: Ground Water;Surface Water

✓ Written to CSV

----------------------------



2026-01-15 10:45:00,919 : INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-15 10:45:00,921 : INFO - Result is {'generated_keywords': 'employment, worker, survey, quarterly, data, sector, india', 'generated_sponsored_keywords': 'worker count, quarterly survey, employment data, sector wise', 'generated_theme': 'Labour and Employment, Employment', 'justification': 'Source columns used: title, sector, sector_resource, note, and ministry_department. The title provides the core topic and time frame (Sector wise ... Fourth Round of Quarterly Employment Survey Jan to March 2022), while the sector field indicates the domain (Employment). sector_resource and note are null, so reliance on sector and title was necessary for keyword generation per the guidelines. Normalization rules applied: convert to lowercase, prefer singular forms (worker instead of workers), avoid punctuation, and avoid internal terms or department acronyms. Semicolon-separated val

TITLE: Sector wise Estimated Number of Workers under Fourth Round of Quarterly Employment Survey from Jan to March 2022
SECTOR: Employment

✓ Written to CSV

----------------------------


✓ Complete! Results saved to results_100.csv


In [20]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock
import csv, json, logging

def process_row_keyword(row):
    """Process a single row - thread-safe"""
    title = row.get("title", "")
    catalog_title = row.get("catalog_title", "")
    ministry_department = row.get("ministry_department", "")
    sector = row.get("sector", "")
    sector_resource = row.get("sector_resource", "")
    cdos_state_ministry = row.get("cdos_state_ministry", "")
    note = row.get("note", "")

    
    # Fallback to 0 if missing / NaN
    raw_hvd = row.get("field_high_value_dataset", 0)
    try:
        hvd_flag = int(raw_hvd) if raw_hvd == raw_hvd else 0  # handles NaN
    except (ValueError, TypeError):
        hvd_flag = 0

    metadata_content = f"""
Title: {title}
Catalog Title: {catalog_title}
Ministry/Department: {ministry_department}
Sector: {sector}
Sector Resource: {sector_resource}
CDOS State Ministry: {cdos_state_ministry}
Note: {note}
HVD Flag: {hvd_flag}
""".strip()

    # LLM must now return JSON including "hvd_category"
    result = get_keywords(metadata_content)  # expected to return a dict (JSON-like)
    logging.info(f"Result is {result}")

    return {
        "title": title,
        "sector": sector,
        "hvd": hvd_flag,
        "metadata_input": metadata_content,
        "llm_response": json.dumps(result, ensure_ascii=False),
        "generated_keywords": result.get("enhanced_keywords", ""),
        "generated_subject": result.get("generated_subject", ""),
        "generated_theme": result.get("generated_theme", ""),
        "justification": result.get("justification", ""),
        "confidence_score": result.get("confidence_score", 0),
        "metadata_gaps": json.dumps(result.get("metadata_gaps", []), ensure_ascii=False),
        # NEW: HVD classification coming from the LLM
        "hvd_category": result.get("hvd_category", ""),
    }

# CSV setup
output_file = "results_100.csv"
fieldnames = [
    "title",
    "sector",
    "hvd",             
    "metadata_input",
    "llm_response",
    "generated_keywords",
    "generated_subject",
    "generated_theme",
    "justification",
    "confidence_score",
    "metadata_gaps",
    "hvd_category",    
]

csv_lock = Lock()
with open(output_file, "w", newline="", encoding="utf-8") as f:
    csv.DictWriter(f, fieldnames=fieldnames).writeheader()

results_array = []
max_workers = 8

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    future_to_row = {
        executor.submit(process_row_keyword, row): idx
        for idx, row in (df.head(6)).iterrows()
    }
    for future in as_completed(future_to_row):
        try:
            result_obj = future.result()
            results_array.append(result_obj)

            # Thread-safe incremental write
            with csv_lock:
                with open(output_file, "a", newline="", encoding="utf-8") as f:
                    csv.DictWriter(f, fieldnames=fieldnames).writerow(result_obj)

            print(
                f"TITLE: {result_obj['title']}\n"
                f"SECTOR: {result_obj['sector']}\n"
                f"HVD: {result_obj['hvd']}\n"
                f"HVD CATEGORY: {result_obj['hvd_category']}\n\nWritten to CSV"
            )
            print("\n----------------------------\n")

        except Exception as e:
            logging.error(f"Row processing failed: {e}")

print(f"\nComplete Results saved to {output_file}")


2025-11-14 14:35:35,337 : INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-14 14:35:35,371 : INFO - Result is {'Title': 'All India Pincode Directory till last month'}


TITLE: All India Pincode Directory till last month
SECTOR: Information and Communications;Post
HVD: 0
HVD CATEGORY: 

Written to CSV

----------------------------



2025-11-14 14:35:35,727 : INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-14 14:35:35,732 : INFO - Result is {'Title': 'Current Daily Price of Various Commodities from Various Markets (Mandi)', 'Enhanced Keywords': 'daily price, commodity prices, mandi prices, market prices, agriculture data, price trends, crop prices, wholesale prices', 'Sponsored Keywords': 'mandi data, current prices, market data, price updates'}


TITLE: Current Daily Price of Various Commodities from Various Markets (Mandi)
SECTOR: Agriculture;Agricultural Marketing
HVD: 1
HVD CATEGORY: 

Written to CSV

----------------------------



2025-11-14 14:35:45,743 : INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-14 14:35:45,747 : INFO - Result is {'error': 'I cannot return JSON in this format.'}


...

TITLE: Registrars of Companies (RoC)-wise Company Master Data
SECTOR: Commerce;Companies
HVD: 1
HVD CATEGORY: 

Written to CSV

----------------------------



2025-11-14 14:35:56,314 : INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-14 14:35:56,320 : INFO - Result is {'Title': 'List of MSME Registered Units under UDYAM', 'Enhanced Keywords': 'registered units, udyam, unit list, business registry, micro units, small units, medium units, enterprise data', 'Sponsored Keywords': 'registered businesses, small business data, micro business units, unit registry, udyam data'}


TITLE: List of MSME Registered Units under UDYAM
SECTOR: Industries;Medium;Micro;Small Scale
HVD: 1
HVD CATEGORY: 

Written to CSV

----------------------------



2025-11-14 14:36:19,936 : INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-14 14:36:19,943 : INFO - Result is {'Title': 'Kisan Call Centre (KCC) - Transcripts of farmers queries & answers', 'Enhanced Keywords': 'farmer queries, kisan, district wise, month wise, agriculture data, farmers welfare, local languages, district data, statistics', 'Sponsored Keywords': 'farmers queries, call centre, transcripts, kisan centre'}


TITLE: Kisan Call Centre (KCC) - Transcripts of farmers queries & answers
SECTOR: Agriculture
HVD: 1
HVD CATEGORY: 

Written to CSV

----------------------------


Complete Results saved to results_100.csv


In [14]:
result_df=pd.read_csv("/home/aakash/NIC/cdl-updated/nic-metadata-cleaning/notebooks/results_100.csv")
result_df.count(axis=0)

title                 100
sector                100
metadata_input        100
llm_response          100
generated_keywords      0
generated_subject       0
generated_theme         0
justification           0
confidence_score      100
metadata_gaps         100
dtype: int64

In [15]:
results_array

[{'title': 'Current Daily Price of Various Commodities from Various Markets (Mandi)',
  'sector': 'Agriculture;Agricultural Marketing',
  'metadata_input': 'Title: Current Daily Price of Various Commodities from Various Markets (Mandi)\nCatalog Title: Current daily price of various commodities from various markets (Mandi)\nMinistry/Department: Ministry of Agriculture and Farmers Welfare;Department of Agriculture and Farmers Welfare;Directorate of Marketing and Inspection (DMI)\nSector: Agriculture;Agricultural Marketing\nSector Resource: Agriculture;Agricultural Marketing\nCDOS State Ministry: Directorate of Marketing and Inspection (DMI)\nNote: nan',
  'llm_response': '{"Title": "Current Daily Price of Various Commodities from Various Markets (Mandi)"}',
  'generated_keywords': '',
  'generated_subject': '',
  'generated_theme': '',
  'justification': '',
  'confidence_score': 0,
  'metadata_gaps': '[]'},
 {'title': 'Registrars of Companies (RoC)-wise Company Master Data',
  'sector':

In [19]:
with open('/home/aakash/NIC/cdl-updated/nic-metadata-cleaning/notebooks/metadata_enrichment_result.json') as f:
    json.dump(results_array, f, indent=2, ensure_ascii=False)

print(f"\n Saved {len(results_array)} results to metadata_enrichment_results.json")

UnsupportedOperation: not writable